# 불확실성 정량화 실습

**Uncertainty Quantification · UQ**

예측값이 얼마나 믿을 만한지 수치로 함께 제시하는 절차.

소재 분야에서 이해하기: 예측 밴드갭에 오차 범위를 붙여 후보를 정렬한다.

이 노트북은 개념을 직접 돌려보기 위한 예제입니다. 데이터는 실제 측정값이 아니라 개념 확인용으로
생성한 값이므로, 결과 수치를 연구 결론으로 쓰지 마세요. 위에서부터 순서대로 실행하세요.
그림의 축 이름은 기본 폰트에 한글 글리프가 없어 영문으로 적었습니다.

참고 자료: [scikit-learn 가우시안 프로세스 문서](https://scikit-learn.org/stable/modules/gaussian_process.html)

## 1. 예측값만으로는 부족합니다

같은 예측값이라도 신뢰도가 다를 수 있습니다. 앙상블로 불확실성을 추정합니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
plt.rcParams['figure.figsize'] = (7, 4)

# 학습 데이터는 온도 650-800 구간에만 있습니다.
x_train = rng.uniform(650, 800, 60)
y_train = 0.02 * (x_train - 650) ** 1.5 + rng.normal(0, 1.0, 60)
x_grid = np.linspace(600, 950, 300)
plt.scatter(x_train, y_train, s=14); plt.xlabel('temperature (C)'); plt.ylabel('property'); plt.show()

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel, WhiteKernel

forest = RandomForestRegressor(n_estimators=300, random_state=0).fit(x_train[:, None], y_train)
tree_predictions = np.array([tree.predict(x_grid[:, None]) for tree in forest.estimators_])
forest_std = tree_predictions.std(0)

gp = GaussianProcessRegressor(kernel=ConstantKernel(1.0) * RBF(40) + WhiteKernel(1.0),
                              normalize_y=True, random_state=0).fit(x_train[:, None], y_train)
gp_mean, gp_std = gp.predict(x_grid[:, None], return_std=True)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.4), sharey=True)
for axis, (name, mean, std) in zip(axes, [('random forest', tree_predictions.mean(0), forest_std),
                                          ('gaussian process', gp_mean, gp_std)]):
    axis.plot(x_grid, mean); axis.fill_between(x_grid, mean - 2 * std, mean + 2 * std, alpha=0.25)
    axis.scatter(x_train, y_train, s=10, c='red'); axis.set_title(name)
    axis.axvspan(650, 800, alpha=0.1, color='green'); axis.set_xlabel('temperature (C)')
plt.tight_layout(); plt.show()

inside = (x_grid > 660) & (x_grid < 790)
outside = x_grid > 860
print('학습 구간 내부 표준편차: forest %.2f / GP %.2f' % (forest_std[inside].mean(), gp_std[inside].mean()))
print('학습 구간 외부 표준편차: forest %.2f / GP %.2f' % (forest_std[outside].mean(), gp_std[outside].mean()))

## 2. 해석

트리 앙상블의 산포는 학습 범위 밖에서도 커지지 않을 수 있습니다. GP는 데이터가 없는 곳에서
불확실성이 자동으로 커집니다. 어떤 불확실성 추정을 쓰는지에 따라 외삽 경고의 신뢰도가 달라집니다.

---

셀의 숫자를 바꿔가며 다시 실행해보면 개념이 더 분명해집니다. 용어 사전으로 돌아가려면
[소재·AI 용어 사전](https://forum.rnddata.org/glossary/#uncertainty-quantification)을 여세요.